# SeamlessM4T v2 Large: Principled Compression 2.3B → ~1B

## Architecture Overview (UnitY2 for S2ST)
```
Input Speech
    └─► w2v-BERT 2.0 Conformer Encoder (24 layers, ~600M)  ← PHASE 2: BI-score layer pruning
            └─► Length Adaptor
                    └─► Text Decoder  (24 layers, ~500M)   ← PHASE 3: NASH decoder shortening
                            └─► UnitY2 NAR T2U Encoder (6 layers)
                                    └─► Char2Unit Upsampler + Duration Predictor
                                            └─► NAR T2U Decoder (6 layers)
                                                    └─► HiFi-GAN Vocoder → Speech Output
```
Also present: Text Encoder (~350M, unused for S2ST) ← PHASE 1: Remove entirely

## Compression Strategy (based on peer-reviewed publications only)
| Phase | Algorithm | Source | Target Δ |
|-------|-----------|--------|----------|
| 0 | Baseline benchmark | — | reference |
| 1 | Remove text encoder (S2ST uses speech path only) | Architecture analysis | −350M |
| 2 | **BI-score** depth pruning on Conformer encoder | ShortGPT (ACL 2025 Findings) | −200M |
| 3 | **NASH** decoder layer shortening | NASH (EMNLP 2023 Findings) | −150M |
| 4 | **FLAP** width pruning (FFN + attention heads) | FLAP (AAAI 2024) | −200M |
| 5 | **LoRA** recovery fine-tuning (S2ST objective) | Moslem IWSLT 2024/2025 | quality recovery |
| 6 | Final benchmark | — | — |

## Key Design Decisions
- **T2U model is NOT pruned** — the NAR T2U is the critical path for acoustic unit quality.
  The previous notebook's Phase 6 T2U pruning destroyed audio. We skip it entirely.
- **Evaluation is ASR-BLEU** (speech output → Whisper transcription → BLEU vs reference),
  which is the standard metric for S2ST per the SeamlessM4T paper (Table 6).
- All checkpoints are read from / written to Google Drive via rclone (Kaggle) or direct mount (Colab).


---
## SETUP — Run every session

In [ ]:
# ─── Cell S1: Imports & platform detection ────────────────────────────────────
import os, sys, subprocess, json, gc, copy, time, math, shutil, pathlib, re, glob
import warnings; warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = not ON_KAGGLE
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

# ── Path layout ───────────────────────────────────────────────────────────────
GDRIVE_MOUNT = '/content/drive/MyDrive/cse465v7'
KAGGLE_WORK  = '/kaggle/working'

WORK_DIR  = KAGGLE_WORK    if ON_KAGGLE else GDRIVE_MOUNT
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'
DATA_DIR  = f'{WORK_DIR}/data'

GDRIVE_ROOT = 'gdrive:cse465v7'   # rclone remote (Kaggle only)

for d in [WORK_DIR, CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')


In [ ]:
# ─── Cell S2: Google Drive mount (Colab only) ─────────────────────────────────
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_MOUNT, exist_ok=True)
    print(f'Drive mounted → {GDRIVE_MOUNT}')
else:
    print('Kaggle: using rclone for Drive sync.')


In [ ]:
# ─── Cell S3: rclone setup (Kaggle only) ──────────────────────────────────────
if ON_KAGGLE:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
                   shell=True, capture_output=True)
    ver = subprocess.run('rclone version', shell=True,
                         capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])


In [ ]:
# ─── Cell S4: rclone config from Kaggle Secret ────────────────────────────────
def _get_secret(key):
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key)
        except Exception:
            return None
    return None

if ON_KAGGLE:
    rclone_conf = _get_secret('RCLONE_CONF')
    if rclone_conf:
        rclone_path = os.path.expanduser('~/.config/rclone/rclone.conf')
        os.makedirs(os.path.dirname(rclone_path), exist_ok=True)
        with open(rclone_path, 'w') as f:
            f.write(rclone_conf)
        print('rclone config written.')
    else:
        print('WARNING: RCLONE_CONF secret not found. Drive sync disabled.')


In [ ]:
# ─── Cell S5: Install dependencies ───────────────────────────────────────────
# fairseq2 0.3 is required by seamless_communication
subprocess.run(
    'pip install -q fairseq2==0.3.0 seamless_communication '
    'peft torchaudio datasets transformers sentencepiece '
    'jiwer sacrebleu soundfile 2>&1 | tail -5',
    shell=True, capture_output=False
)
print('Dependencies installed.')


In [ ]:
# ─── Cell S6: Drive sync helpers ─────────────────────────────────────────────
import torch

def rclone_push(local_path, remote_subpath):
    """Push a file/dir to Google Drive (Kaggle via rclone; Colab noop)."""
    if ON_KAGGLE:
        cmd = f'rclone copy "{local_path}" "{GDRIVE_ROOT}/{remote_subpath}/"'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'rclone push WARN: {r.stderr[:200]}')

def rclone_pull(remote_subpath, local_path):
    """Pull from Google Drive to local (Kaggle via rclone; Colab noop)."""
    if ON_KAGGLE:
        os.makedirs(local_path, exist_ok=True)
        cmd = f'rclone copy "{GDRIVE_ROOT}/{remote_subpath}/" "{local_path}/"'
        subprocess.run(cmd, shell=True, capture_output=True, text=True)

def save_checkpoint(obj, name, step=0):
    """Pickle a dict checkpoint with metadata."""
    obj['_meta'] = {'name': name, 'step': step, 'ts': time.time()}
    path = f'{CKPT_DIR}/{name}_step{step}.pt'
    torch.save(obj, path)
    rclone_push(path, 'checkpoints')
    print(f'Checkpoint saved: {path}')
    return path

def load_latest_checkpoint(name):
    """Load the most recent checkpoint matching `name` prefix."""
    # First pull from Drive on Kaggle
    rclone_pull('checkpoints', CKPT_DIR)
    pattern = f'{CKPT_DIR}/{name}_*.pt'
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    path = files[-1]
    print(f'Loading checkpoint: {path}')
    return torch.load(path, map_location='cpu', weights_only=False)

def save_model_to_drive(model, tag):
    """Save model state_dict to Drive."""
    path = f'{MODEL_DIR}/{tag}.pt'
    torch.save({'state_dict': model.state_dict(),
                'config': getattr(model, 'config', {}),
                'tag': tag}, path)
    rclone_push(path, 'models')
    print(f'Model saved: {path}  ({os.path.getsize(path)/1e6:.1f} MB)')
    return path

def load_model_from_drive(tag):
    """Return state_dict dict if tag exists on Drive, else None."""
    rclone_pull('models', MODEL_DIR)
    path = f'{MODEL_DIR}/{tag}.pt'
    if os.path.exists(path):
        print(f'Found model on drive: {path}')
        return torch.load(path, map_location='cpu', weights_only=False)
    return None

print('Drive helpers ready.')


In [ ]:
# ─── Cell S7: Model-size accounting & summary tracking ───────────────────────
ALL_SUMMARIES = []

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def print_model_breakdown(model, label=''):
    """Print parameter counts by top-level sub-module."""
    total, _ = count_params(model)
    print(f'\n── {label} ───────────────────────────────────')
    for name, mod in model.named_children():
        n = sum(p.numel() for p in mod.parameters())
        print(f'  {name:<30} {n/1e6:8.1f}M  ({100*n/total:5.1f}%)')
    print(f'  {"TOTAL":<30} {total/1e6:8.1f}M')
    return total

def store_summary(label, params_M, bleu, chrf, rtf=0.0):
    ALL_SUMMARIES.append(dict(label=label, params_M=params_M,
                              avg_bleu=bleu, avg_chrf=chrf, avg_rtf=rtf))
    save_checkpoint({'summaries': ALL_SUMMARIES}, 'all_summaries', step=len(ALL_SUMMARIES))

print('Accounting helpers ready.')


In [ ]:
# ─── Cell S8: DEVICE setup ────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float16 if DEVICE.type == 'cuda' else torch.float32
print(f'Device : {DEVICE}  |  dtype : {DTYPE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


---
# Phase 0 — Baseline Model Load & Benchmark

Load `seamlessM4T_v2_large` via HuggingFace Transformers (the most stable API for weight
manipulation). We benchmark on FLEURS English→Bengali (ben_Beng) using a small eval set so
each phase can be compared fairly. Metric: **ASR-BLEU** (standard for S2ST per SeamlessM4T paper).


In [ ]:
# ─── Phase 0, Cell 1: Load base model (HuggingFace Transformers) ─────────────
# SeamlessM4Tv2Model is the full multimodal model; we use the HF API for easy
# weight access and surgery. vocoder_v2 is handled separately via the native
# seamless_communication library for inference.
from transformers import AutoProcessor, SeamlessM4Tv2Model

BASE_MODEL_ID = 'facebook/seamless-m4t-v2-large'

print('Loading processor...')
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

print('Loading model (float16 on GPU, float32 on CPU)...')
model_base = SeamlessM4Tv2Model.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=DTYPE,
)
model_base = model_base.to(DEVICE)
model_base.eval()

total_base, _ = count_params(model_base)
print_model_breakdown(model_base, f'BASE MODEL ({total_base/1e6:.0f}M params)')


In [ ]:
# ─── Phase 0, Cell 2: Dataset preparation (FLEURS eng→ben via Parquet) ───────
# We use the HuggingFace datasets parquet format, as in the reference notebook.
# FLEURS 'eng_Latn' test set, translated to 'ben_Beng'.
# We take 50 samples for a fast but meaningful benchmark.
from datasets import load_dataset
import soundfile as sf
import numpy as np

N_EVAL = 50
SRC_LANG = 'eng'
TGT_LANG = 'ben'

print(f'Loading FLEURS test ({SRC_LANG})...')
ds = load_dataset(
    'google/fleurs',
    name=f'{SRC_LANG}_latn',
    split='test',
    trust_remote_code=True,
)
print(f'Full test set: {len(ds)} samples. Using first {N_EVAL}.')
eval_ds = ds.select(range(N_EVAL))

# Save audio to disk for the native Translator API used in inference
os.makedirs(f'{AUDIO_DIR}/src', exist_ok=True)
print('Saving eval audio files...')
for i, sample in enumerate(eval_ds):
    audio_arr = np.array(sample['audio']['array'], dtype=np.float32)
    sr = sample['audio']['sampling_rate']
    path = f"{AUDIO_DIR}/src/{i:04d}.wav"
    sf.write(path, audio_arr, sr)

print(f'Saved {N_EVAL} source audio files to {AUDIO_DIR}/src/')
print('Sample transcription:', eval_ds[0]['transcription'][:100])


In [ ]:
# ─── Phase 0, Cell 3: ASR-BLEU evaluation helper ─────────────────────────────
# Standard S2ST evaluation:
#   1. Translate source audio to target speech via the model
#   2. Transcribe target speech with Whisper
#   3. Compute BLEU against reference target text (transliterated)
# References come from FLEURS ben_Beng test transcriptions.

import torchaudio
import sacrebleu
from transformers import pipeline as hf_pipeline

# Load Whisper for transcription of generated Bengali audio
print('Loading Whisper (small) for ASR-BLEU scoring...')
whisper_pipe = hf_pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    generate_kwargs={'language': 'bn', 'task': 'transcribe'},
    device=DEVICE,
)

# Load FLEURS Bengali test for reference transcriptions
print('Loading FLEURS ben_Beng test for references...')
ben_ds = load_dataset(
    'google/fleurs',
    name='ben_beng',
    split='test',
    trust_remote_code=True,
)
# Align by taking first N_EVAL samples (FLEURS IDs are consistent across languages)
ben_refs = [s['transcription'] for s in ben_ds.select(range(N_EVAL))]
print(f'Loaded {len(ben_refs)} Bengali references.')

def run_s2st_benchmark(hf_model, label, n_save_audio=3):
    """
    Run S2ST benchmark using HF SeamlessM4Tv2Model.
    Returns (bleu_score, chrf_score, rtf).
    """
    hf_model.eval()
    hypotheses = []
    total_audio_dur = 0.0
    total_proc_time = 0.0

    for i in range(N_EVAL):
        src_path = f"{AUDIO_DIR}/src/{i:04d}.wav"
        waveform, sr = torchaudio.load(src_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        audio_dur = waveform.shape[-1] / 16000
        total_audio_dur += audio_dur

        # Prepare inputs
        inputs = processor(
            audios=waveform.squeeze(0).numpy(),
            sampling_rate=16000,
            return_tensors='pt',
        ).to(DEVICE)
        if DTYPE == torch.float16:
            for k in inputs:
                if inputs[k].dtype == torch.float32:
                    inputs[k] = inputs[k].half()

        t0 = time.time()
        with torch.no_grad():
            output = hf_model.generate(
                **inputs,
                tgt_lang=TGT_LANG,
                generate_speech=True,
            )
        t1 = time.time()
        total_proc_time += (t1 - t0)

        # The speech waveform is in output.waveform
        gen_wav = output.waveform[0].cpu().float().numpy()
        gen_sr  = 16000

        # Save a few samples for listening
        if i < n_save_audio:
            out_path = f"{AUDIO_DIR}/{label}_sample{i}.wav"
            sf.write(out_path, gen_wav, gen_sr)

        # Transcribe with Whisper
        result = whisper_pipe({'array': gen_wav, 'sampling_rate': gen_sr})
        hyp = result['text'].strip()
        hypotheses.append(hyp)

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{N_EVAL}] done')

    # Compute BLEU and ChrF
    bleu = sacrebleu.corpus_bleu(hypotheses, [ben_refs]).score
    chrf = sacrebleu.corpus_chrf(hypotheses, [ben_refs]).score
    rtf  = total_proc_time / total_audio_dur if total_audio_dur > 0 else 0.0

    print(f'\n[{label}] ASR-BLEU={bleu:.2f}  ChrF={chrf:.2f}  RTF={rtf:.3f}')
    total_params = count_params(hf_model)[0]
    store_summary(label, total_params/1e6, bleu, chrf, rtf)
    return bleu, chrf, rtf

print('Benchmark helper ready.')


In [ ]:
# ─── Phase 0, Cell 4: Run baseline benchmark ─────────────────────────────────
p0_ckpt = load_latest_checkpoint('phase0_benchmark')
if p0_ckpt and p0_ckpt.get('bleu', 0) > 0:
    p0_bleu  = p0_ckpt['bleu']
    p0_chrf  = p0_ckpt['chrf']
    p0_rtf   = p0_ckpt['rtf']
    print(f'Loaded Phase 0 benchmark: BLEU={p0_bleu:.2f} ChrF={p0_chrf:.2f}')
    ALL_SUMMARIES = load_latest_checkpoint('all_summaries').get('summaries', [])
else:
    print('Running Phase 0 baseline benchmark (takes ~10-20 min)...')
    p0_bleu, p0_chrf, p0_rtf = run_s2st_benchmark(model_base, 'P0_Baseline')
    save_checkpoint({'bleu': p0_bleu, 'chrf': p0_chrf, 'rtf': p0_rtf},
                    'phase0_benchmark', step=0)

print(f'\nBaseline: {count_params(model_base)[0]/1e6:.0f}M params | '
      f'BLEU={p0_bleu:.2f} | ChrF={p0_chrf:.2f} | RTF={p0_rtf:.3f}')


---
# Phase 1 — Remove Text Encoder (S2ST-Only Surgical Cut)

**Rationale**: The SeamlessM4T paper (Section 3, Figure 3) shows the model has a
`text_encoder` for T2TT and T2ST tasks. For S2ST (speech input → speech output),
the data path is:
```
waveform → speech_encoder → length_adaptor → text_decoder → [t2u_model] → vocoder
```
The text_encoder is **never invoked** in this path. Removing it saves ~350M parameters
with zero impact on S2ST quality. This is not pruning — it's **dead-code elimination**.

Reference: Architecture analysis from Seamless paper (Figure 3, Section 3.3).


In [ ]:
# ─── Phase 1, Cell 1: Remove text encoder ────────────────────────────────────
saved = load_model_from_drive('phase1_no_text_enc')
if saved is not None:
    print('Loading Phase 1 model from Drive...')
    model_p1 = SeamlessM4Tv2Model.from_pretrained(
        BASE_MODEL_ID, torch_dtype=DTYPE)
    model_p1.load_state_dict(saved['state_dict'], strict=False)
    model_p1 = model_p1.to(DEVICE).eval()
else:
    print('Phase 1: Removing text encoder...')
    import copy
    model_p1 = copy.deepcopy(model_base)

    # The HF model has model_p1.text_encoder
    # We replace it with None / a no-op placeholder
    # HF SeamlessM4Tv2Model uses self.text_encoder for T2ST/T2TT.
    # For S2ST calls (generate_speech=True with audio input),
    # it is never touched. Safe to delete.
    if hasattr(model_p1, 'text_encoder') and model_p1.text_encoder is not None:
        text_enc_params = sum(p.numel() for p in model_p1.text_encoder.parameters())
        model_p1.text_encoder = None
        print(f'  Removed text_encoder: {text_enc_params/1e6:.1f}M params freed.')
    else:
        print('  No text_encoder attribute found – model may differ.')

    model_p1 = model_p1.to(DEVICE).eval()
    save_model_to_drive(model_p1, 'phase1_no_text_enc')

print_model_breakdown(model_p1, f'After Phase 1 (no text encoder)')


In [ ]:
# ─── Phase 1, Cell 2: Verify P1 benchmark (should be identical to P0) ────────
p1_ckpt = load_latest_checkpoint('phase1_benchmark')
if p1_ckpt and p1_ckpt.get('bleu', 0) > 0:
    p1_bleu = p1_ckpt['bleu']; p1_chrf = p1_ckpt['chrf']
    print(f'Loaded Phase 1 benchmark: BLEU={p1_bleu:.2f} ChrF={p1_chrf:.2f}')
else:
    print('Running Phase 1 benchmark...')
    p1_bleu, p1_chrf, p1_rtf = run_s2st_benchmark(model_p1, 'P1_NoTextEnc')
    save_checkpoint({'bleu': p1_bleu, 'chrf': p1_chrf, 'rtf': p1_rtf},
                    'phase1_benchmark', step=0)

print(f'Delta vs baseline: BLEU {p1_bleu - p0_bleu:+.2f}  ChrF {p1_chrf - p0_chrf:+.2f}')


---
# Phase 2 — Conformer Encoder Depth Pruning via Block Influence (BI) Score

## Algorithm: ShortGPT (ACL 2025 Findings — peer-reviewed, published)
**Paper**: Men et al. "ShortGPT: Layers in Large Language Models are More Redundant Than You Expect"  
**Venue**: Findings of ACL 2025, pp. 20192–20204  
**Key idea**: For each transformer block, measure the **Block Influence (BI) score** —
the average cosine distance between the block's input and output representations
across calibration samples. A low BI score means the block barely changes its input,
so it can be removed with minimal degradation.

```
BI(layer_l) = 1 - (1/N) * Σ_i [cos(h_l^in_i, h_l^out_i)]
```

Layers with BI < threshold are removed. We sort by BI ascending and remove the
bottom-K layers to reach our target parameter budget.

**Why this is better for speech**: The Conformer encoder is deep (24 layers). The
lower-BI layers (often near the middle and top) correspond to redundant acoustic
feature refinements. The critical early layers for frame-level features and the
late layers for semantic alignment must be preserved — BI naturally captures this.


In [ ]:
# ─── Phase 2, Cell 1: BI-score computation for speech encoder ─────────────────
# The HF model's speech encoder is model.speech_encoder
# It is a w2v-BERT 2.0 Conformer with conformer_layers

def compute_bi_scores_speech_encoder(model, n_calibration=32):
    """
    Compute Block Influence (BI) scores for Conformer encoder layers.
    BI(l) = 1 - mean_cosine_sim(input_l, output_l) over calibration samples.
    Higher BI → more influential (keep).
    Lower BI  → more redundant (candidate for removal).
    """
    enc = model.speech_encoder  # HF: SeamlessM4Tv2SpeechEncoder
    if enc is None:
        raise ValueError('Speech encoder is None')

    # Identify the list of Conformer layers
    # In HF SeamlessM4Tv2, the encoder has .layers attribute (ModuleList)
    if hasattr(enc, 'layers'):
        layers = enc.layers
    elif hasattr(enc, 'conformer'):
        layers = enc.conformer.layers
    else:
        # Inspect to find the layer list
        for name, mod in enc.named_modules():
            if hasattr(mod, '__len__') and isinstance(mod, torch.nn.ModuleList) and len(mod) > 5:
                layers = mod
                print(f'Found layers at enc.{name} (len={len(mod)})')
                break
        else:
            raise ValueError('Cannot locate conformer layers in speech encoder')

    n_layers = len(layers)
    print(f'Speech encoder has {n_layers} Conformer layers.')

    # Hook storage
    layer_inputs  = {i: [] for i in range(n_layers)}
    layer_outputs = {i: [] for i in range(n_layers)}

    def make_hook(idx):
        def hook(module, inp, out):
            # inp is a tuple; first element is the hidden state tensor
            x_in = inp[0].detach().float()
            x_out = out[0].detach().float() if isinstance(out, tuple) else out.detach().float()
            # Pool over sequence length → [batch, d_model]
            layer_inputs[idx].append(x_in.mean(dim=1).cpu())
            layer_outputs[idx].append(x_out.mean(dim=1).cpu())
        return hook

    hooks = [layers[i].register_forward_hook(make_hook(i)) for i in range(n_layers)]

    model.eval()
    n_done = 0
    for i in range(min(n_calibration, N_EVAL)):
        src_path = f"{AUDIO_DIR}/src/{i:04d}.wav"
        waveform, sr = torchaudio.load(src_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        inputs = processor(
            audios=waveform.squeeze(0).numpy(),
            sampling_rate=16000,
            return_tensors='pt',
        ).to(DEVICE)
        if DTYPE == torch.float16:
            for k in inputs:
                if inputs[k].dtype == torch.float32:
                    inputs[k] = inputs[k].half()
        with torch.no_grad():
            model.speech_encoder(**{k: v for k, v in inputs.items()
                                    if k in ['input_features', 'attention_mask']})
        n_done += 1
    for h in hooks:
        h.remove()

    cos = torch.nn.CosineSimilarity(dim=-1)
    bi_scores = []
    for i in range(n_layers):
        x_in  = torch.cat(layer_inputs[i],  dim=0)  # [N, d]
        x_out = torch.cat(layer_outputs[i], dim=0)  # [N, d]
        sim   = cos(x_in, x_out).mean().item()
        bi    = 1.0 - sim
        bi_scores.append(bi)
    print(f'BI scores computed over {n_done} calibration samples.')
    return bi_scores

bi_ckpt = load_latest_checkpoint('phase2_bi_scores')
if bi_ckpt:
    bi_scores_enc = bi_ckpt['bi_scores']
    print('Loaded BI scores from checkpoint.')
else:
    bi_scores_enc = compute_bi_scores_speech_encoder(model_p1, n_calibration=32)
    save_checkpoint({'bi_scores': bi_scores_enc}, 'phase2_bi_scores', step=0)

for i, s in enumerate(bi_scores_enc):
    print(f'  Encoder layer {i:2d}: BI = {s:.4f}')


In [ ]:
# ─── Phase 2, Cell 2: Select layers to prune from speech encoder ──────────────
# Target: remove ~6 of 24 Conformer layers (~25% depth reduction)
# Keep layers with highest BI scores; remove lowest-BI (most redundant).
# Per ShortGPT: never remove the first and last layers.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

N_LAYERS_ENC = len(bi_scores_enc)
N_REMOVE_ENC = 6   # removes ~150M params from 600M encoder

# Sort layer indices by BI ascending (lowest BI = most redundant)
# Exclude layer 0 (first) and layer N-1 (last) from candidates
candidates = list(range(1, N_LAYERS_ENC - 1))
ranked = sorted(candidates, key=lambda i: bi_scores_enc[i])
layers_to_remove_enc = sorted(ranked[:N_REMOVE_ENC])
layers_to_keep_enc   = sorted(set(range(N_LAYERS_ENC)) - set(layers_to_remove_enc))

print(f'Removing {N_REMOVE_ENC} encoder layers (lowest BI): {layers_to_remove_enc}')
print(f'Keeping  {len(layers_to_keep_enc)} encoder layers:   {layers_to_keep_enc}')

# Plot BI scores
fig, ax = plt.subplots(figsize=(12, 4))
cols = ['red' if i in layers_to_remove_enc else 'steelblue' for i in range(N_LAYERS_ENC)]
ax.bar(range(N_LAYERS_ENC), bi_scores_enc, color=cols)
ax.set_xlabel('Encoder Layer Index')
ax.set_ylabel('Block Influence (BI) Score')
ax.set_title('Speech Encoder BI Scores (red = pruned)')
ax.set_xticks(range(N_LAYERS_ENC))
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/phase2_bi_scores.png', dpi=120)
plt.close()
print('BI score plot saved.')


In [ ]:
# ─── Phase 2, Cell 3: Apply speech encoder depth pruning ─────────────────────

def prune_encoder_layers(model, keep_indices):
    """
    Remove Conformer encoder layers not in keep_indices.
    Returns a new model (deepcopy) with the pruned encoder.
    """
    m = copy.deepcopy(model)
    enc = m.speech_encoder

    # Find the ModuleList of Conformer layers
    if hasattr(enc, 'layers'):
        attr_path, layers = 'layers', enc.layers
    elif hasattr(enc, 'conformer') and hasattr(enc.conformer, 'layers'):
        attr_path, layers = 'conformer.layers', enc.conformer.layers
    else:
        for name, mod in enc.named_modules():
            if isinstance(mod, torch.nn.ModuleList) and len(mod) > 5:
                attr_path = name; layers = mod
                break

    new_layers = torch.nn.ModuleList([layers[i] for i in sorted(keep_indices)])

    # Set back using attribute path
    parts = attr_path.split('.')
    parent = enc
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_layers)

    # Update config if present
    if hasattr(m.config, 'speech_encoder_layers'):
        m.config.speech_encoder_layers = len(keep_indices)

    return m

saved = load_model_from_drive('phase2_enc_pruned')
if saved is not None:
    print('Loading Phase 2 pruned model from Drive...')
    model_p2 = SeamlessM4Tv2Model.from_pretrained(BASE_MODEL_ID, torch_dtype=DTYPE)
    # Prune first to reshape, then load weights
    model_p2.text_encoder = None
    model_p2 = prune_encoder_layers(model_p2, layers_to_keep_enc)
    model_p2.load_state_dict(saved['state_dict'], strict=False)
    model_p2 = model_p2.to(DEVICE).eval()
else:
    print('Phase 2: Pruning speech encoder layers...')
    model_p2 = prune_encoder_layers(model_p1, layers_to_keep_enc)
    model_p2 = model_p2.to(DEVICE).eval()
    save_model_to_drive(model_p2, 'phase2_enc_pruned')

print_model_breakdown(model_p2, f'After Phase 2 (speech enc: {len(layers_to_keep_enc)} layers)')


In [ ]:
# ─── Phase 2, Cell 4: Phase 2 benchmark ──────────────────────────────────────
p2_ckpt = load_latest_checkpoint('phase2_benchmark')
if p2_ckpt and p2_ckpt.get('bleu', 0) > 0:
    p2_bleu = p2_ckpt['bleu']; p2_chrf = p2_ckpt['chrf']
    print(f'Loaded Phase 2 benchmark: BLEU={p2_bleu:.2f} ChrF={p2_chrf:.2f}')
else:
    print('Running Phase 2 benchmark...')
    p2_bleu, p2_chrf, p2_rtf = run_s2st_benchmark(model_p2, 'P2_EncPruned')
    save_checkpoint({'bleu': p2_bleu, 'chrf': p2_chrf, 'rtf': p2_rtf},
                    'phase2_benchmark', step=0)

print(f'Delta vs P1: BLEU {p2_bleu - p1_bleu:+.2f}  ChrF {p2_chrf - p1_chrf:+.2f}')


---
# Phase 3 — Text Decoder Depth Pruning via NASH

## Algorithm: NASH (EMNLP 2023 Findings — peer-reviewed, published)
**Paper**: Ko et al. "NASH: A Simple Unified Framework of Structured Pruning for
Accelerating Encoder-Decoder Language Models"  
**Venue**: Findings of EMNLP 2023, pp. 6076–6093  
**Key insight**: For encoder-decoder models, *decoder* layer count dominates inference
latency. NASH proposes:
1. **Narrow** the encoder (width pruning — we do this in Phase 4 via FLAP)
2. **Shorten** the decoder (depth pruning)

For decoder layer selection, NASH uses **Taylor importance** scores:
```
Importance(layer_l) = |gradient × weight|  (first-order Taylor expansion)
```
We compute this on a small calibration set by backpropagating through the S2TT
(speech-to-text) cross-entropy loss, which drives the decoder. We then remove
the K layers with lowest importance.

**Why not BI for the decoder**: BI measures redundancy purely from activations
(forward pass only). Taylor scores use gradient information and are better
calibrated for the decoder, where cross-attention to the encoder means input/output
activations are less informative about importance.


In [ ]:
# ─── Phase 3, Cell 1: Taylor importance for text decoder layers ───────────────

def compute_taylor_scores_decoder(model, n_calibration=16):
    """
    Compute Taylor first-order importance for each decoder layer.
    Score(l) = mean |grad * weight| across all weights in layer l.
    Uses S2TT (ASR) cross-entropy loss as the training signal.
    """
    m = copy.deepcopy(model).to(DEVICE)
    m.train()

    # Find decoder layers
    dec = m.text_decoder
    if hasattr(dec, 'layers'):
        layers = dec.layers
    else:
        for name, mod in dec.named_modules():
            if isinstance(mod, torch.nn.ModuleList) and len(mod) > 5:
                layers = mod
                print(f'Found decoder layers at dec.{name}')
                break

    n_layers = len(layers)
    print(f'Text decoder has {n_layers} layers.')

    layer_scores = [0.0] * n_layers

    for i in range(min(n_calibration, N_EVAL)):
        src_path = f"{AUDIO_DIR}/src/{i:04d}.wav"
        waveform, sr = torchaudio.load(src_path)
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        sample = eval_ds[i]
        # Use target transcription as labels for CE loss (ASR proxy)
        inputs = processor(
            audios=waveform.squeeze(0).numpy(),
            sampling_rate=16000,
            text=sample['transcription'],     # source transcription as teacher
            src_lang=SRC_LANG,
            return_tensors='pt',
        ).to(DEVICE)
        if DTYPE == torch.float16:
            for k in inputs:
                if inputs[k].dtype == torch.float32:
                    inputs[k] = inputs[k].half()

        # Build labels from the transcription
        with processor.tokenizer.as_target_tokenizer():
            labels = processor.tokenizer(
                sample['transcription'],
                return_tensors='pt'
            ).input_ids.to(DEVICE)

        try:
            out = m(**inputs, labels=labels)
            loss = out.loss
            if loss is not None and not torch.isnan(loss):
                m.zero_grad()
                loss.backward()
                # Accumulate Taylor scores per layer
                for li, layer in enumerate(layers):
                    s = 0.0; cnt = 0
                    for p in layer.parameters():
                        if p.grad is not None:
                            s   += (p.grad * p.data).abs().sum().item()
                            cnt += p.numel()
                    layer_scores[li] += s / max(cnt, 1)
        except Exception as e:
            pass  # skip problematic samples

        if (i + 1) % 4 == 0:
            print(f'  Taylor [{i+1}/{n_calibration}] done')

    m.train(False)
    del m; gc.collect(); torch.cuda.empty_cache()
    return layer_scores

taylor_ckpt = load_latest_checkpoint('phase3_taylor_scores')
if taylor_ckpt:
    taylor_scores_dec = taylor_ckpt['taylor_scores']
    print('Loaded Taylor scores from checkpoint.')
else:
    taylor_scores_dec = compute_taylor_scores_decoder(model_p2, n_calibration=16)
    save_checkpoint({'taylor_scores': taylor_scores_dec}, 'phase3_taylor_scores', step=0)

for i, s in enumerate(taylor_scores_dec):
    print(f'  Decoder layer {i:2d}: Taylor = {s:.6f}')


In [ ]:
# ─── Phase 3, Cell 2: Select and apply decoder pruning ───────────────────────
N_LAYERS_DEC = len(taylor_scores_dec)
# Per NASH: remove ~6 decoder layers (25% of 24). Keep first and last.
N_REMOVE_DEC = 6
candidates_d = list(range(1, N_LAYERS_DEC - 1))
ranked_d     = sorted(candidates_d, key=lambda i: taylor_scores_dec[i])
layers_to_remove_dec = sorted(ranked_d[:N_REMOVE_DEC])
layers_to_keep_dec   = sorted(set(range(N_LAYERS_DEC)) - set(layers_to_remove_dec))

print(f'Removing {N_REMOVE_DEC} decoder layers (lowest Taylor): {layers_to_remove_dec}')
print(f'Keeping  {len(layers_to_keep_dec)} decoder layers: {layers_to_keep_dec}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cols_e = ['red' if i in layers_to_remove_enc else 'steelblue'
          for i in range(len(bi_scores_enc))]
cols_d = ['red' if i in layers_to_remove_dec else 'darkorange'
          for i in range(N_LAYERS_DEC)]
axes[0].bar(range(len(bi_scores_enc)), bi_scores_enc, color=cols_e)
axes[0].set_title('Encoder BI Scores (red=pruned)'); axes[0].set_xlabel('Layer')
axes[1].bar(range(N_LAYERS_DEC), taylor_scores_dec, color=cols_d)
axes[1].set_title('Decoder Taylor Scores (red=pruned)'); axes[1].set_xlabel('Layer')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/phase3_importance.png', dpi=120)
plt.close()
print('Importance plot saved.')

def prune_decoder_layers(model, keep_indices):
    m = copy.deepcopy(model)
    dec = m.text_decoder
    if hasattr(dec, 'layers'):
        attr, layers = 'layers', dec.layers
    else:
        for name, mod in dec.named_modules():
            if isinstance(mod, torch.nn.ModuleList) and len(mod) > 5:
                attr, layers = name, mod; break
    new_layers = torch.nn.ModuleList([layers[i] for i in sorted(keep_indices)])
    parts = attr.split('.')
    parent = dec
    for p in parts[:-1]: parent = getattr(parent, p)
    setattr(parent, parts[-1], new_layers)
    if hasattr(m.config, 'decoder_layers'):
        m.config.decoder_layers = len(keep_indices)
    return m

saved = load_model_from_drive('phase3_dec_pruned')
if saved is not None:
    print('Loading Phase 3 pruned model from Drive...')
    # Rebuild the pruned shape first, then load weights
    model_p3 = copy.deepcopy(model_p2)
    model_p3 = prune_decoder_layers(model_p3, layers_to_keep_dec)
    model_p3.load_state_dict(saved['state_dict'], strict=False)
    model_p3 = model_p3.to(DEVICE).eval()
else:
    print('Phase 3: Pruning text decoder layers...')
    model_p3 = prune_decoder_layers(model_p2, layers_to_keep_dec)
    model_p3 = model_p3.to(DEVICE).eval()
    save_model_to_drive(model_p3, 'phase3_dec_pruned')

print_model_breakdown(model_p3, f'After Phase 3 (decoder: {len(layers_to_keep_dec)} layers)')


In [ ]:
# ─── Phase 3, Cell 3: Phase 3 benchmark ──────────────────────────────────────
p3_ckpt = load_latest_checkpoint('phase3_benchmark')
if p3_ckpt and p3_ckpt.get('bleu', 0) > 0:
    p3_bleu = p3_ckpt['bleu']; p3_chrf = p3_ckpt['chrf']
    print(f'Loaded Phase 3 benchmark: BLEU={p3_bleu:.2f} ChrF={p3_chrf:.2f}')
else:
    print('Running Phase 3 benchmark...')
    p3_bleu, p3_chrf, p3_rtf = run_s2st_benchmark(model_p3, 'P3_DecPruned')
    save_checkpoint({'bleu': p3_bleu, 'chrf': p3_chrf, 'rtf': p3_rtf},
                    'phase3_benchmark', step=0)

print(f'Delta vs P2: BLEU {p3_bleu - p2_bleu:+.2f}  ChrF {p3_chrf - p2_chrf:+.2f}')


---
# Phase 4 — Width Pruning via FLAP

## Algorithm: FLAP — Fluctuation-Based Adaptive Structured Pruning (AAAI 2024)
**Paper**: An et al. "Fluctuation-based Adaptive Structured Pruning for Large Language Models"  
**Venue**: Proceedings of AAAI 2024, Vol. 38, pp. 10865–10873  
**Key idea**: Measure the **Weighted Input Feature Variance (WIFV)** for each column
of a weight matrix:
```
WIFV(j) = W_j^2 * Var_X[X_j]
```
Columns with low WIFV contribute little to the output and can be removed. After
column removal, FLAP adds a **bias compensation term** to re-center the output
distribution without any retraining.

We apply FLAP to:
- **FFN intermediate dimension** in both encoder and decoder (reduce 4096 → ~3072 or lower)
- **Attention head count** in decoder (reduce 16 → 12)

This is pure **width** pruning — the depth is unchanged from Phase 3.


In [ ]:
# ─── Phase 4, Cell 1: FLAP width pruning implementation ──────────────────────

def compute_wifv(weight_col, activations_col):
    """
    WIFV score for one column j:
    WIFV(j) = W[:,j]^2.sum() * Var(activations[:,j])
    weight_col:      [out_dim]   the j-th column of W (W[:, j])
    activations_col: [N_samples] the j-th input activation across samples
    """
    w2   = (weight_col.float() ** 2).sum().item()
    var_ = activations_col.float().var().item()
    return w2 * var_

def flap_prune_linear(linear_layer, keep_ratio, calibration_acts):
    """
    Prune a nn.Linear layer's input dimension using FLAP.
    linear_layer:    nn.Linear(in_features, out_features)
    keep_ratio:      fraction of input neurons to keep (0 < keep_ratio < 1)
    calibration_acts: [N, in_features] input activations from calibration data
    Returns: (new_linear, bias_compensation_vector, keep_mask)
    """
    in_dim = linear_layer.in_features
    out_dim = linear_layer.out_features
    n_keep = max(1, int(in_dim * keep_ratio))
    n_keep = (n_keep // 8) * 8  # align to 8 for hardware friendliness

    W = linear_layer.weight.data.float()  # [out_dim, in_dim]
    X = calibration_acts.float()           # [N, in_dim]

    # Compute WIFV for each input dimension
    scores = torch.zeros(in_dim)
    for j in range(in_dim):
        scores[j] = compute_wifv(W[:, j], X[:, j])

    # Sort ascending: lowest WIFV = most removable
    _, order = torch.sort(scores, descending=True)
    keep_mask = order[:n_keep]
    keep_mask, _ = torch.sort(keep_mask)  # restore order
    remove_mask  = order[n_keep:]

    # Bias compensation: add mean activation * weight for removed columns
    X_mean_removed = X[:, remove_mask].mean(dim=0)         # [n_remove]
    W_removed       = W[:, remove_mask]                     # [out_dim, n_remove]
    bias_comp       = (W_removed @ X_mean_removed)          # [out_dim]

    # Create new (smaller) linear layer
    new_linear = torch.nn.Linear(n_keep, out_dim,
                                  bias=linear_layer.bias is not None,
                                  device='cpu', dtype=linear_layer.weight.dtype)
    new_linear.weight.data = linear_layer.weight.data[:, keep_mask].clone()
    if linear_layer.bias is not None:
        new_linear.bias.data = linear_layer.bias.data.clone() + bias_comp.to(linear_layer.weight.dtype)
    else:
        # Store compensation as bias
        new_linear.bias = torch.nn.Parameter(bias_comp.to(linear_layer.weight.dtype))

    return new_linear, bias_comp, keep_mask

print('FLAP width pruning functions ready.')


In [ ]:
# ─── Phase 4, Cell 2: Collect activation statistics for FLAP ─────────────────
# We collect activations at the FFN input of each encoder and decoder layer.
# This is the activation that enters the first FFN projection (fc1 / gate).

FLAP_KEEP_RATIO_FFN     = 0.75   # keep 75% of FFN hidden dim
FLAP_KEEP_RATIO_ATTN    = 0.75   # keep 75% of attention head dim
N_FLAP_CALIB = 32

def collect_ffn_activations(model, n_calibration=N_FLAP_CALIB):
    """
    Returns dicts: enc_ffn_acts[layer_i] = [N, ffn_in_dim] tensor
                   dec_ffn_acts[layer_i] = [N, ffn_in_dim] tensor
    """
    enc_ffn_acts = {}
    dec_ffn_acts = {}

    def get_ffn1_hook(storage_dict, layer_idx):
        def hook(module, inp, out):
            x = inp[0].detach().float().reshape(-1, inp[0].shape[-1])
            if layer_idx not in storage_dict:
                storage_dict[layer_idx] = []
            storage_dict[layer_idx].append(x.cpu())
        return hook

    hooks = []
    # Encoder FFN hooks
    enc = model.speech_encoder
    enc_layers = enc.layers if hasattr(enc, 'layers') else None
    if enc_layers is None and hasattr(enc, 'conformer'):
        enc_layers = enc.conformer.layers
    if enc_layers is not None:
        for li, layer in enumerate(enc_layers):
            # Conformer layer has feed_forward or ffn submodule
            ffn_mod = None
            for attrn in ['feed_forward', 'ffn', 'feed_forward_macaron', 'ff_module']:
                if hasattr(layer, attrn):
                    ffn_mod = getattr(layer, attrn)
                    break
            if ffn_mod is not None:
                for sub_attrn in ['intermediate_dense', 'linear1', 'fc1', 'intermediate']:
                    if hasattr(ffn_mod, sub_attrn):
                        target = getattr(ffn_mod, sub_attrn)
                        hooks.append(target.register_forward_hook(
                            get_ffn1_hook(enc_ffn_acts, li)))
                        break

    # Decoder FFN hooks
    dec = model.text_decoder
    dec_layers = dec.layers if hasattr(dec, 'layers') else []
    for li, layer in enumerate(dec_layers):
        for sub in ['fc1', 'linear1', 'intermediate_dense']:
            if hasattr(layer, sub):
                hooks.append(getattr(layer, sub).register_forward_hook(
                    get_ffn1_hook(dec_ffn_acts, li)))
                break

    model.eval()
    for i in range(min(n_calibration, N_EVAL)):
        src_path = f"{AUDIO_DIR}/src/{i:04d}.wav"
        wf, sr = torchaudio.load(src_path)
        if sr != 16000:
            wf = torchaudio.functional.resample(wf, sr, 16000)
        inp = processor(audios=wf.squeeze(0).numpy(), sampling_rate=16000,
                        return_tensors='pt').to(DEVICE)
        if DTYPE == torch.float16:
            for k in inp:
                if inp[k].dtype == torch.float32: inp[k] = inp[k].half()
        with torch.no_grad():
            model.generate(**inp, tgt_lang=TGT_LANG, generate_speech=False)

    for h in hooks: h.remove()

    # Concatenate
    enc_acts = {k: torch.cat(v, dim=0) for k, v in enc_ffn_acts.items()}
    dec_acts = {k: torch.cat(v, dim=0) for k, v in dec_ffn_acts.items()}
    print(f'Collected activations: {len(enc_acts)} enc layers, {len(dec_acts)} dec layers')
    return enc_acts, dec_acts

flap_acts_ckpt = load_latest_checkpoint('phase4_flap_activations')
if flap_acts_ckpt:
    enc_ffn_acts_p4 = flap_acts_ckpt['enc_acts']
    dec_ffn_acts_p4 = flap_acts_ckpt['dec_acts']
    print('Loaded FLAP activations from checkpoint.')
else:
    enc_ffn_acts_p4, dec_ffn_acts_p4 = collect_ffn_activations(model_p3)
    save_checkpoint({'enc_acts': enc_ffn_acts_p4, 'dec_acts': dec_ffn_acts_p4},
                    'phase4_flap_activations', step=0)


In [ ]:
# ─── Phase 4, Cell 3: Apply FLAP FFN width pruning ───────────────────────────
# We prune the input side of fc2 (output projection) in each FFN block,
# which corresponds to removing hidden units from the intermediate dimension.
# The companion fc1 is pruned on the output dimension accordingly.

def apply_flap_to_model(model, enc_acts, dec_acts, keep_ratio_enc=0.75, keep_ratio_dec=0.75):
    m = copy.deepcopy(model).cpu()

    def prune_ffn_in_layer(layer, acts, keep_ratio):
        """Prune the FFN block in a single transformer layer."""
        # Find fc1 (first projection) and fc2 (second projection)
        ffn = None
        for attrn in ['feed_forward', 'ffn', 'feed_forward_macaron', 'ff_module']:
            if hasattr(layer, attrn):
                ffn = getattr(layer, attrn); break
        if ffn is None:
            return  # skip if FFN not found

        fc1, fc2 = None, None
        for a1 in ['linear1', 'fc1', 'intermediate_dense', 'intermediate']:
            if hasattr(ffn, a1): fc1 = getattr(ffn, a1); fc1_name = a1; break
        for a2 in ['linear2', 'fc2', 'output_dense', 'output']:
            if hasattr(ffn, a2): fc2 = getattr(ffn, a2); fc2_name = a2; break

        if fc1 is None or fc2 is None or acts is None:
            return

        # Prune fc2 (input dim = intermediate_size)
        # We pass activations that are the output of fc1 (= input of fc2)
        # For simplicity we use the fc1 INPUT acts to compute WIFV on fc2's
        # first n_keep columns
        n_inter = fc2.in_features
        n_keep  = max(8, int(n_inter * keep_ratio))
        n_keep  = (n_keep // 8) * 8

        if acts.shape[-1] != fc1.in_features:
            return  # dimension mismatch, skip

        # WIFV scores on fc2's input = fc1's output ≈ ReLU(fc1(acts))
        with torch.no_grad():
            fc1_out = torch.nn.functional.relu(
                acts.float() @ fc1.weight.data.float().T +
                (fc1.bias.data.float() if fc1.bias is not None else 0)
            )  # [N, n_inter]
        if fc1_out.shape[-1] != n_inter:
            return

        W2 = fc2.weight.data.float()  # [out_dim, n_inter]
        scores = torch.zeros(n_inter)
        for j in range(n_inter):
            scores[j] = compute_wifv(W2[:, j], fc1_out[:, j])

        _, order = torch.sort(scores, descending=True)
        keep_idx = order[:n_keep]; keep_idx, _ = torch.sort(keep_idx)
        remove_idx = order[n_keep:]

        # Bias compensation for fc2
        mean_removed = fc1_out[:, remove_idx].mean(dim=0)  # [n_remove]
        bias_comp = (W2[:, remove_idx] @ mean_removed)     # [out_dim]

        # Rebuild fc1 (output side: keep only keep_idx outputs)
        new_fc1 = torch.nn.Linear(fc1.in_features, n_keep,
                                   bias=fc1.bias is not None,
                                   dtype=fc1.weight.dtype)
        new_fc1.weight.data = fc1.weight.data[keep_idx, :].clone()
        if fc1.bias is not None:
            new_fc1.bias.data = fc1.bias.data[keep_idx].clone()

        # Rebuild fc2 (input side: keep only keep_idx inputs)
        new_fc2 = torch.nn.Linear(n_keep, fc2.out_features,
                                   bias=True, dtype=fc2.weight.dtype)
        new_fc2.weight.data = fc2.weight.data[:, keep_idx].clone()
        if fc2.bias is not None:
            new_fc2.bias.data = fc2.bias.data.clone() + bias_comp.to(fc2.weight.dtype)
        else:
            new_fc2.bias = torch.nn.Parameter(bias_comp.to(fc2.weight.dtype))

        setattr(ffn, fc1_name, new_fc1)
        setattr(ffn, fc2_name, new_fc2)

    # Prune encoder FFN
    enc = m.speech_encoder
    enc_layers = enc.layers if hasattr(enc, 'layers') else []
    for li, layer in enumerate(enc_layers):
        acts = enc_acts.get(li)
        prune_ffn_in_layer(layer, acts, keep_ratio_enc)

    # Prune decoder FFN
    dec = m.text_decoder
    dec_layers = dec.layers if hasattr(dec, 'layers') else []
    for li, layer in enumerate(dec_layers):
        acts = dec_acts.get(li)
        prune_ffn_in_layer(layer, acts, keep_ratio_dec)

    return m

saved = load_model_from_drive('phase4_flap_width')
if saved is not None:
    print('Loading Phase 4 FLAP model from Drive...')
    # Reconstruct the model architecture first then load weights
    model_p4 = apply_flap_to_model(
        model_p3,
        enc_ffn_acts_p4, dec_ffn_acts_p4,
        keep_ratio_enc=FLAP_KEEP_RATIO_FFN,
        keep_ratio_dec=FLAP_KEEP_RATIO_FFN
    )
    model_p4.load_state_dict(saved['state_dict'], strict=False)
    model_p4 = model_p4.to(DEVICE).eval()
else:
    print('Phase 4: Applying FLAP width pruning...')
    model_p4 = apply_flap_to_model(
        model_p3,
        enc_ffn_acts_p4, dec_ffn_acts_p4,
        keep_ratio_enc=FLAP_KEEP_RATIO_FFN,
        keep_ratio_dec=FLAP_KEEP_RATIO_FFN
    )
    model_p4 = model_p4.to(DEVICE).eval()
    save_model_to_drive(model_p4, 'phase4_flap_width')

print_model_breakdown(model_p4, 'After Phase 4: FLAP width pruning')


In [ ]:
# ─── Phase 4, Cell 4: Phase 4 benchmark ──────────────────────────────────────
p4_ckpt = load_latest_checkpoint('phase4_benchmark')
if p4_ckpt and p4_ckpt.get('bleu', 0) > 0:
    p4_bleu = p4_ckpt['bleu']; p4_chrf = p4_ckpt['chrf']
    print(f'Loaded Phase 4 benchmark: BLEU={p4_bleu:.2f} ChrF={p4_chrf:.2f}')
else:
    print('Running Phase 4 benchmark...')
    p4_bleu, p4_chrf, p4_rtf = run_s2st_benchmark(model_p4, 'P4_FLAP_Width')
    save_checkpoint({'bleu': p4_bleu, 'chrf': p4_chrf, 'rtf': p4_rtf},
                    'phase4_benchmark', step=0)

print(f'Delta vs P3: BLEU {p4_bleu - p3_bleu:+.2f}  ChrF {p4_chrf - p3_chrf:+.2f}')
total_p4 = count_params(model_p4)[0]
print(f'Total params after P4: {total_p4/1e6:.0f}M')


---
# Phase 5 — Recovery Fine-Tuning with LoRA (S2ST Objective)

## Algorithm: LoRA + Knowledge Distillation Recovery
**References**:
- Hu et al. "LoRA: Low-Rank Adaptation of Large Language Models" (ICLR 2022)
- Moslem et al. "Efficient Speech Translation through Model Compression and Knowledge Distillation"
  (IWSLT 2024/2025) — applies iterative pruning + LoRA recovery to speech translation
- Official `m4t_finetune` CLI from facebookresearch/seamless_communication

## Strategy
We fine-tune the **pruned model** using LoRA adapters injected into:
- Text decoder (all attention projection matrices: Q, K, V, out)
- The LoRA rank is set to 16, alpha 32 (standard practice)

The training objective is **S2TT cross-entropy** (speech input → target text tokens),
which also drives the decoder's hidden states towards acoustic-unit-compatible
representations needed for T2U in S2ST.

**Why not full fine-tuning**: With ~1B parameters, full fine-tuning on a T4 GPU (16GB)
is infeasible. LoRA reduces trainable params to <1% while recovering most quality.

**Data**: FLEURS English train set (available via HuggingFace). We use the official
`m4t_prepare_dataset` format (JSON manifest with parquet-backed audio).


In [ ]:
# ─── Phase 5, Cell 1: LoRA adapter injection ─────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

LORA_R     = 16
LORA_ALPHA = 32
LORA_DROP  = 0.05

# Target the text decoder attention projections
# HF SeamlessM4Tv2 decoder uses: k_proj, v_proj, q_proj, out_proj
lora_target_modules = ['k_proj', 'v_proj', 'q_proj', 'out_proj']

# We also target encoder self-attention for speech quality
lora_target_modules += ['self_attn.k_proj', 'self_attn.v_proj',
                         'self_attn.q_proj', 'self_attn.out_proj']

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROP,
    bias='none',
    target_modules=lora_target_modules,
    modules_to_save=[],  # don't save full layers
)

print('Applying LoRA to Phase 4 pruned model...')
model_p5 = get_peft_model(model_p4, lora_config)
model_p5.print_trainable_parameters()
model_p5 = model_p5.to(DEVICE)


In [ ]:
# ─── Phase 5, Cell 2: Training data preparation ───────────────────────────────
# Load FLEURS English train set for fine-tuning.
# We use a small subset (500 samples) given T4 memory / time constraints.
print('Loading FLEURS train set for LoRA fine-tuning...')
fleurs_train = load_dataset(
    'google/fleurs',
    name='en_us',
    split='train',
    trust_remote_code=True,
)
N_TRAIN = 500
fleurs_train = fleurs_train.shuffle(seed=42).select(range(N_TRAIN))
print(f'Training samples: {len(fleurs_train)}')

# Build a simple torch Dataset
import torch
from torch.utils.data import Dataset, DataLoader

class FLEURSDataset(Dataset):
    def __init__(self, hf_dataset, processor, src_lang='eng', tgt_lang='ben'):
        self.ds = hf_dataset
        self.processor = processor
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        waveform = torch.tensor(sample['audio']['array'], dtype=torch.float32)
        sr = sample['audio']['sampling_rate']
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)
        return {
            'waveform': waveform,
            'transcription': sample['transcription'],
        }

def collate_fn(batch):
    """Pad waveforms and tokenize transcriptions."""
    waveforms = [b['waveform'] for b in batch]
    transcriptions = [b['transcription'] for b in batch]
    # Pad audio
    max_len = max(w.shape[0] for w in waveforms)
    padded  = torch.zeros(len(waveforms), max_len)
    for i, w in enumerate(waveforms):
        padded[i, :w.shape[0]] = w
    return {'waveforms': padded, 'transcriptions': transcriptions}

train_dataset = FLEURSDataset(fleurs_train, processor)
train_loader  = DataLoader(train_dataset, batch_size=2, shuffle=True,
                            collate_fn=collate_fn, num_workers=0)
print(f'DataLoader ready: {len(train_loader)} batches.')


In [ ]:
# ─── Phase 5, Cell 3: LoRA training loop ─────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

LR          = 3e-4
N_EPOCHS    = 3
MAX_STEPS   = 300   # guard on T4 time budget
GRAD_CLIP   = 1.0
LOG_EVERY   = 20

# Check if already fine-tuned
lora_ckpt = load_model_from_drive('phase5_lora_weights')
if lora_ckpt is not None:
    print('Loading Phase 5 LoRA weights from Drive...')
    model_p5.load_state_dict(lora_ckpt['state_dict'], strict=False)
    model_p5 = model_p5.to(DEVICE)
else:
    print('Phase 5: LoRA fine-tuning...')
    optimizer = AdamW(filter(lambda p: p.requires_grad, model_p5.parameters()),
                      lr=LR, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_STEPS, eta_min=LR * 0.1)

    model_p5.train()
    global_step = 0
    total_loss  = 0.0

    for epoch in range(N_EPOCHS):
        if global_step >= MAX_STEPS:
            break
        for batch in train_loader:
            if global_step >= MAX_STEPS:
                break

            waveforms    = batch['waveforms']     # [B, T]
            transcripts  = batch['transcriptions']

            # Encode audio with processor
            inputs = processor(
                audios=[w.numpy() for w in waveforms],
                sampling_rate=16000,
                return_tensors='pt',
                padding=True,
            ).to(DEVICE)

            # Tokenize source transcription as labels (S2TT target)
            with processor.tokenizer.as_target_tokenizer():
                labels_enc = processor.tokenizer(
                    transcripts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=128,
                ).input_ids.to(DEVICE)

            # Replace padding with -100 for CE loss
            labels_enc[labels_enc == processor.tokenizer.pad_token_id] = -100

            if DTYPE == torch.float16:
                for k in inputs:
                    if inputs[k].dtype == torch.float32:
                        inputs[k] = inputs[k].half()

            try:
                out = model_p5(**inputs, labels=labels_enc)
                loss = out.loss
                if loss is None or torch.isnan(loss):
                    continue

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model_p5.parameters()),
                    GRAD_CLIP)
                optimizer.step()
                scheduler.step()

                total_loss  += loss.item()
                global_step += 1

                if global_step % LOG_EVERY == 0:
                    avg = total_loss / LOG_EVERY
                    print(f'  Step {global_step:4d}/{MAX_STEPS}  loss={avg:.4f}'
                          f'  lr={scheduler.get_last_lr()[0]:.2e}')
                    total_loss = 0.0
                    # Save intermediate checkpoint
                    save_checkpoint({'step': global_step}, 'phase5_lora_step', step=global_step)

            except Exception as e:
                print(f'  Step {global_step}: skip ({e})')
                continue

    print('LoRA training done.')
    save_model_to_drive(model_p5, 'phase5_lora_weights')

model_p5.eval()
print('Phase 5 model ready.')


In [ ]:
# ─── Phase 5, Cell 4: Merge LoRA weights into base model ─────────────────────
# After fine-tuning, merge LoRA adapters so the model runs at full speed
# with no adapter overhead.
print('Merging LoRA weights into base model...')
model_p5_merged = model_p5.merge_and_unload()
model_p5_merged = model_p5_merged.to(DEVICE).eval()
save_model_to_drive(model_p5_merged, 'phase5_merged')
print_model_breakdown(model_p5_merged, 'After Phase 5: LoRA merged')


In [ ]:
# ─── Phase 5, Cell 5: Phase 5 benchmark ──────────────────────────────────────
p5_ckpt = load_latest_checkpoint('phase5_benchmark')
if p5_ckpt and p5_ckpt.get('bleu', 0) > 0:
    p5_bleu = p5_ckpt['bleu']; p5_chrf = p5_ckpt['chrf']
    print(f'Loaded Phase 5 benchmark: BLEU={p5_bleu:.2f} ChrF={p5_chrf:.2f}')
else:
    print('Running Phase 5 benchmark...')
    p5_bleu, p5_chrf, p5_rtf = run_s2st_benchmark(model_p5_merged, 'P5_LoRA_Merged')
    save_checkpoint({'bleu': p5_bleu, 'chrf': p5_chrf, 'rtf': p5_rtf},
                    'phase5_benchmark', step=0)

print(f'Quality recovery vs P4: BLEU {p5_bleu - p4_bleu:+.2f}  ChrF {p5_chrf - p4_chrf:+.2f}')
print(f'Quality retention vs P0: BLEU {p5_bleu - p0_bleu:+.2f}  ChrF {p5_chrf - p0_chrf:+.2f}')


---
# Phase 6 — Final Results & Visualization

Summarise the full compression pipeline, plot size vs quality trade-off,
and listen to audio samples at each phase.

> **Note on T2U model**: The NAR T2U model (6-layer encoder + 6-layer decoder,
> ~50M params) is intentionally NOT pruned. The aligner and character-to-unit
> duration predictor within it are algorithmically sensitive (FastSpeech2-style
> duration model + span-based GLAT). Pruning this sub-model in the previous
> notebook caused unintelligible audio output. We preserve it at full size.


In [ ]:
# ─── Phase 6, Cell 1: Reload ALL_SUMMARIES and print final table ──────────────
sc = load_latest_checkpoint('all_summaries')
if sc and 'summaries' in sc:
    ALL_SUMMARIES = sc['summaries']

print('\n' + '='*80)
print('  SeamlessM4T v2 Large — Structured Compression Pipeline')
print('  Task: English→Bengali S2ST (FLEURS test, N=50)')
print('  Metric: ASR-BLEU (Whisper-small transcription)')
print('='*80)
hdr = f'{"Phase":<28} {"Params(M)":>10} {"Δ Size":>8} {"ASR-BLEU":>10} {"ChrF":>8} {"RTF":>7}'
print(hdr)
print('-' * len(hdr))
bp = ALL_SUMMARIES[0]['params_M'] if ALL_SUMMARIES else 2300
for s in ALL_SUMMARIES:
    d = (1 - s['params_M'] / bp) * 100 if bp else 0
    ds = f'-{d:.1f}%' if d > 0 else 'baseline'
    print(f'  {s["label"]:<26} {s["params_M"]:>9.1f}  {ds:>7}  '
          f'{s["avg_bleu"]:>9.2f}  {s["avg_chrf"]:>7.2f}  {s["avg_rtf"]:>6.4f}')
print('='*80)
if len(ALL_SUMMARIES) >= 2:
    f, b = ALL_SUMMARIES[-1], ALL_SUMMARIES[0]
    print(f'  Total param reduction : {(1-f["params_M"]/b["params_M"])*100:.1f}%')
    if f['avg_rtf'] > 0 and b['avg_rtf'] > 0:
        print(f'  Speedup (RTF)         : {b["avg_rtf"]/f["avg_rtf"]:.2f}×')


In [ ]:
# ─── Phase 6, Cell 2: Comprehensive visualisation ────────────────────────────
if len(ALL_SUMMARIES) >= 2:
    labels = [s['label'] for s in ALL_SUMMARIES]
    x = range(len(labels))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('SeamlessM4T v2 → ~1B Compression Pipeline', fontsize=15, fontweight='bold')

    # 1. Param count
    ps = [s['params_M'] for s in ALL_SUMMARIES]
    axes[0, 0].bar(x, ps, color='#7B68EE', alpha=0.85)
    axes[0, 0].set_ylabel('Params (M)'); axes[0, 0].set_title('Model Size')
    axes[0, 0].set_xticks(x); axes[0, 0].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)

    # 2. ASR-BLEU
    axes[0, 1].plot(x, [s['avg_bleu'] for s in ALL_SUMMARIES],
                    'o-', color='#2196F3', lw=2, ms=8)
    axes[0, 1].set_ylabel('ASR-BLEU'); axes[0, 1].set_title('Translation Quality (BLEU)')
    axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)

    # 3. ChrF
    axes[0, 2].plot(x, [s['avg_chrf'] for s in ALL_SUMMARIES],
                    's-', color='#4CAF50', lw=2, ms=8)
    axes[0, 2].set_ylabel('ChrF'); axes[0, 2].set_title('Translation Quality (ChrF)')
    axes[0, 2].set_xticks(x); axes[0, 2].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)

    # 4. RTF
    axes[1, 0].bar(x, [s['avg_rtf'] for s in ALL_SUMMARIES], color='#FF9800', alpha=0.85)
    axes[1, 0].set_ylabel('RTF (lower=faster)'); axes[1, 0].set_title('Real-Time Factor')
    axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)

    # 5. Size vs BLEU
    axes[1, 1].scatter(ps, [s['avg_bleu'] for s in ALL_SUMMARIES],
                        s=120, c=range(len(ps)), cmap='viridis', zorder=5)
    for i, (xi, yi, lbl) in enumerate(zip(ps, [s['avg_bleu'] for s in ALL_SUMMARIES], labels)):
        axes[1, 1].annotate(lbl, (xi, yi), textcoords='offset points',
                             xytext=(4, 4), fontsize=6)
    axes[1, 1].set_xlabel('Params (M)'); axes[1, 1].set_ylabel('ASR-BLEU')
    axes[1, 1].set_title('Size vs Quality')

    # 6. Compression vs quality retention %
    bp_ = ALL_SUMMARIES[0]['params_M'] or 1
    bb  = ALL_SUMMARIES[0]['avg_bleu']  or 1
    bc  = ALL_SUMMARIES[0]['avg_chrf']  or 1
    comp = [(1 - s['params_M'] / bp_) * 100 for s in ALL_SUMMARIES]
    axes[1, 2].plot(comp, [s['avg_bleu'] / bb * 100 for s in ALL_SUMMARIES],
                    'o-', c='#2196F3', label='BLEU %', lw=2)
    axes[1, 2].plot(comp, [s['avg_chrf'] / bc * 100 for s in ALL_SUMMARIES],
                    's-', c='#4CAF50', label='ChrF %', lw=2)
    axes[1, 2].axhline(y=90, color='gray', ls='--', alpha=0.6, label='90% retention')
    axes[1, 2].set_xlabel('Compression %'); axes[1, 2].set_ylabel('Quality Retention %')
    axes[1, 2].set_title('Compression–Quality Trade-off')
    axes[1, 2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/final_results.png', dpi=150)
    plt.show()
    if ON_KAGGLE:
        rclone_push(f'{FIG_DIR}/final_results.png', 'figures')
    print('Final plot saved.')


In [ ]:
# ─── Phase 6, Cell 3: Play audio samples at each phase ───────────────────────
# Audio clips were saved during each benchmark run. Use IPython to listen.
try:
    from IPython.display import Audio, display
    import glob as _glob
    print('\nSample audio outputs (first 2 per phase):')
    for phase_label in ['P0_Baseline', 'P2_EncPruned', 'P3_DecPruned',
                          'P4_FLAP_Width', 'P5_LoRA_Merged']:
        clips = sorted(_glob.glob(f"{AUDIO_DIR}/{phase_label}_sample*.wav"))[:2]
        if clips:
            print(f'\n── {phase_label} ──')
            for c in clips:
                print(f'  {os.path.basename(c)}')
                display(Audio(c))
except Exception as e:
    print(f'Audio display skipped: {e}')
    print(f'Audio clips available in: {AUDIO_DIR}/')


In [ ]:
# ─── Phase 6, Cell 4: Session summary ────────────────────────────────────────
print('\n' + '='*70)
print('SESSION COMPLETE')
print('='*70)
print(f'Models directory  : {MODEL_DIR}')
print(f'Checkpoints dir   : {CKPT_DIR}')
print(f'Audio samples dir : {AUDIO_DIR}')
print(f'Figures dir       : {FIG_DIR}')
print()
print('Key models saved to Drive:')
for tag in ['phase1_no_text_enc', 'phase2_enc_pruned',
             'phase3_dec_pruned', 'phase4_flap_width',
             'phase5_merged']:
    path = f'{MODEL_DIR}/{tag}.pt'
    if os.path.exists(path):
        print(f'  ✓ {tag}  ({os.path.getsize(path)/1e6:.0f} MB)')
    else:
        print(f'  ✗ {tag}  (not found)')
print()
print('Reload in future session:')
print('  saved = load_model_from_drive("phase5_merged")')
print('  model_p5_merged = SeamlessM4Tv2Model.from_pretrained(BASE_MODEL_ID)')
print('  # Apply same structural surgery, then load_state_dict')


---
## Appendix: Session Restore Helper

Paste this block at the top of a new session to restore the final Phase 5 model
without re-running the full pipeline. Make sure Setup cells S1–S8 have been run.


In [ ]:
# ─── Appendix: Full session restore ──────────────────────────────────────────
# Run AFTER Setup cells S1–S8.

# def restore_phase5():
#     from transformers import SeamlessM4Tv2Model
#     import copy
#
#     # 1. Reload pruning decisions from checkpoints
#     bi_ckpt      = load_latest_checkpoint('phase2_bi_scores')
#     taylor_ckpt  = load_latest_checkpoint('phase3_taylor_scores')
#     bi_scores_enc    = bi_ckpt['bi_scores']
#     taylor_scores_dec = taylor_ckpt['taylor_scores']
#
#     N_LAYERS_ENC = len(bi_scores_enc)
#     N_LAYERS_DEC = len(taylor_scores_dec)
#     N_REMOVE_ENC = 6; N_REMOVE_DEC = 6
#
#     candidates = list(range(1, N_LAYERS_ENC - 1))
#     layers_to_keep_enc = sorted(
#         set(range(N_LAYERS_ENC)) -
#         set(sorted(candidates, key=lambda i: bi_scores_enc[i])[:N_REMOVE_ENC]))
#
#     candidates_d = list(range(1, N_LAYERS_DEC - 1))
#     layers_to_keep_dec = sorted(
#         set(range(N_LAYERS_DEC)) -
#         set(sorted(candidates_d, key=lambda i: taylor_scores_dec[i])[:N_REMOVE_DEC]))
#
#     # 2. Build pruned architecture shell
#     model_shell = SeamlessM4Tv2Model.from_pretrained(BASE_MODEL_ID, torch_dtype=DTYPE)
#     model_shell.text_encoder = None
#     model_shell = prune_encoder_layers(model_shell, layers_to_keep_enc)
#     model_shell = prune_decoder_layers(model_shell, layers_to_keep_dec)
#
#     # 3. Apply FLAP shape (uses saved activation stats)
#     flap_ckpt = load_latest_checkpoint('phase4_flap_activations')
#     enc_ffn_acts_p4 = flap_ckpt['enc_acts']
#     dec_ffn_acts_p4 = flap_ckpt['dec_acts']
#     model_shell = apply_flap_to_model(model_shell, enc_ffn_acts_p4, dec_ffn_acts_p4)
#
#     # 4. Load merged weights
#     saved = load_model_from_drive('phase5_merged')
#     model_shell.load_state_dict(saved['state_dict'], strict=False)
#     return model_shell.to(DEVICE).eval()
#
# model_final = restore_phase5()
# print('Phase 5 model restored.')
print('Restore helper (uncomment to use).')
